# 03 – Evaluation of all methods (Sophia)

Compares the reconstructions of **NeRF**, **3DGS** (and U-Net, once available) with the
ground truth (full-view FBP, 984 views) and with the **FBP baseline** on exactly the same slices.

**Steps**
1. Inventory: which output files exist, what size/format they have.
2. Match every output PNG to its ground-truth slice (by series and slice number).
3. Bring all images onto the same footing (same size, same intensity scale).
4. Compute PSNR / SSIM / RMSE with `metrics.py` (whole image + head only).
5. Summary table (mean ± SD per method), comparison figure, CSV for the report.

**Fairness rule:** the outputs are 8-bit PNGs. So the FBP baseline is passed through
*exactly the same PNG treatment* (min-max → 8 bit) before it is scored. Otherwise FBP would
have an unfair advantage from being stored as float.

## 0. Setup

In [ ]:
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    if not os.path.exists("/content/IPPR_CT_Reconstruction"):
        !git clone -q https://github.com/gs0232/IPPR_CT_Reconstruction.git /content/IPPR_CT_Reconstruction
    sys.path.insert(0, "/content/IPPR_CT_Reconstruction/Baselines")
else:
    sys.path.insert(0, ".")

In [ ]:
import re, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from skimage.transform import iradon, resize

import metrics
metrics._self_test()

## 1. Configuration

In [ ]:
DATA_ROOT   = Path("/content/drive/MyDrive/MGST_5_IPPR_CT_Reconstruction")   # <- adjust if needed
OUTPUT_ROOT = DATA_ROOT / "outputs"                 # outputs/nerf, outputs/3dgs, outputs/unet
GT_ROOT     = DATA_ROOT / "gt_fullview_npy"         # gt_fullview_npy/series_XXXX/slice_XX.npy
SPARSE_ROOT = DATA_ROOT / "sparse_v098_npy"         # sparse sinograms + theta_deg.npy (for the FBP baseline)
EVAL_DIR    = DATA_ROOT / "evaluation_results"
EVAL_DIR.mkdir(exist_ok=True)

METHODS     = ["nerf", "3dgs", "unet"]              # folders inside outputs/ (missing ones are skipped)
N_VIEWS     = 98                                    # sparse level the methods started from
IMG_SIZE    = 888
N_ANGLES    = 984
THETA_FULL  = np.linspace(0, 360, N_ANGLES, endpoint=False)

## 2. Inventory – what did the others deliver?
Run this first and look at the output: number of files, file names, image size and mode.

In [ ]:
def find_id(path):
    # Extract (series, slice) from any naming style, e.g.
    #   outputs/nerf/series_1009/slice_08.png  or  outputs/nerf/series_1009_slice08.png
    s = str(path)
    ser = re.search(r"series_?(\d+)", s)
    sl  = re.search(r"slice_?(\d+)", Path(s).name) or re.search(r"slice_?(\d+)", s)
    if not ser or not sl:
        return None
    return f"series_{ser.group(1)}", int(sl.group(1))

files = {}
for m in METHODS:
    folder = OUTPUT_ROOT / m
    found = sorted(folder.rglob("*.png")) + sorted(folder.rglob("*.npy")) if folder.exists() else []
    if not found:
        print(f"{m:5s}: no files found in {folder} – skipped")
        continue
    files[m] = found
    ids = [find_id(f) for f in found]
    n_bad = sum(i is None for i in ids)
    first = found[0]
    arr = np.load(first) if first.suffix == ".npy" else np.array(Image.open(first))
    mode = "npy" if first.suffix == ".npy" else Image.open(first).mode
    print(f"{m:5s}: {len(found)} files | e.g. {first.relative_to(OUTPUT_ROOT)} | shape {arr.shape} | "
          f"mode {mode} | dtype {arr.dtype} | min/max {arr.min()}/{arr.max()}")
    print(f"       series: {sorted({i[0] for i in ids if i})}")
    if n_bad:
        print(f"       WARNING: {n_bad} file names without series/slice number")

## 3. Loading and bringing everything onto the same footing

In [ ]:
def load_any(path):
    # PNG (gray/RGB/RGBA, 8 or 16 bit) or .npy -> 2D float32 array
    if Path(path).suffix == ".npy":
        a = np.load(path).astype(np.float32)
    else:
        a = np.array(Image.open(path)).astype(np.float32)
        if a.ndim == 3:
            a = a[..., 0]                       # grey PNG saved as RGB(A): channels are identical
    return a

def to_image(a, theta=None):
    # If the array is a sinogram (detectors x angles), reconstruct it with FBP; otherwise return as is.
    if a.shape == (IMG_SIZE, IMG_SIZE):
        return a
    if a.shape[0] == IMG_SIZE and theta is not None and a.shape[1] == len(theta):
        return iradon(a, theta=theta, filter_name="ramp", output_size=IMG_SIZE, circle=True).astype(np.float32)
    if a.shape == (IMG_SIZE, N_ANGLES):
        return iradon(a, theta=THETA_FULL, filter_name="ramp", output_size=IMG_SIZE, circle=True).astype(np.float32)
    # other image size (e.g. 512x512) -> resize to 888x888 so it lines up with the ground truth
    return resize(a, (IMG_SIZE, IMG_SIZE), order=1, preserve_range=True, anti_aliasing=True).astype(np.float32)

def minmax(a):
    return (a - a.min()) / (a.max() - a.min() + 1e-12)

def as_png(a):
    # Simulate the PNG export the others used: stretch to 0-1, round to 8 bit
    return np.round(minmax(a) * 255) / 255

def match_intensity(pred, gt):
    # PNGs have lost the original intensity scale. Find the best a*pred + b that matches gt
    # (least squares). This only fixes brightness/contrast – it cannot remove artefacts.
    a, b = np.polyfit(pred.ravel(), gt.ravel(), 1)
    return a * pred + b

def load_gt(series, sl):
    p = GT_ROOT / series / f"slice_{sl:02d}.npy"
    return to_image(np.load(p).astype(np.float32)) if p.exists() else None

THETA_SPARSE = np.load(SPARSE_ROOT / "theta_deg.npy")

def fbp_baseline(series, sl):
    # Sparse-view FBP of the same slice, from the saved sparse sinogram (or image)
    p = SPARSE_ROOT / series / f"slice_{sl:02d}.npy"
    return to_image(np.load(p).astype(np.float32), THETA_SPARSE) if p.exists() else None

## 4. Visual check of one slice (do the images line up?)
If a method looks **mirrored, rotated or shifted** compared with the ground truth, the metrics
will be meaningless – fix that first (see the `ORIENT` dictionary below).

In [ ]:
# If a method's images are flipped/rotated, set e.g. ORIENT["nerf"] = np.flipud  or  np.rot90
ORIENT = {m: (lambda x: x) for m in METHODS}

common = None
for m, fl in files.items():
    ids = {find_id(f) for f in fl if find_id(f)}
    common = ids if common is None else common & ids
common = sorted(i for i in (common or []) if load_gt(*i) is not None)
print(len(common), "slices are available for ALL methods and have a ground truth")

series, sl = common[len(common) // 2]
gt = minmax(load_gt(series, sl))
panels = {"ground truth": gt, f"FBP {N_VIEWS} views": match_intensity(as_png(fbp_baseline(series, sl)), gt)}
for m, fl in files.items():
    f = next(f for f in fl if find_id(f) == (series, sl))
    panels[m] = match_intensity(ORIENT[m](to_image(load_any(f), THETA_SPARSE)), gt)

fig, ax = plt.subplots(2, len(panels), figsize=(3.6 * len(panels), 7.4))
for j, (name, img) in enumerate(panels.items()):
    ax[0, j].imshow(img, cmap="gray", vmin=0, vmax=1); ax[0, j].set_title(f"{name}\n{series} slice {sl}", fontsize=9)
    ax[1, j].imshow(np.abs(img - gt), cmap="magma", vmin=0, vmax=0.3); ax[1, j].set_title("|error|", fontsize=9)
for a in ax.ravel(): a.axis("off")
plt.tight_layout(); plt.savefig(EVAL_DIR / "comparison_example.png", dpi=150); plt.show()

## 5. Metrics for all slices
Every method (and FBP) is scored on the **same slices**: those delivered by all methods.

In [ ]:
rows = []
t0 = time.perf_counter()
for series, sl in common:
    gt = minmax(load_gt(series, sl))                          # ground truth scaled to 0-1
    preds = {"FBP": as_png(fbp_baseline(series, sl))}         # FBP with the same PNG treatment
    for m, fl in files.items():
        f = next(f for f in fl if find_id(f) == (series, sl))
        preds[m] = ORIENT[m](to_image(load_any(f), THETA_SPARSE))
    for name, pred in preds.items():
        pred = match_intensity(pred, gt)
        rows.append({"method": name, "series": series, "slice": sl, "views": N_VIEWS,
                     **metrics.evaluate(pred, gt, already_normalised=True)})
res = pd.DataFrame(rows)
res.to_csv(EVAL_DIR / "evaluation_all_slices.csv", index=False)
print(f"{len(common)} slices x {res.method.nunique()} methods in {time.perf_counter() - t0:.0f} s")

## 6. Summary table and plot

In [ ]:
cols = ["psnr", "ssim", "rmse", "psnr_head", "ssim_head", "rmse_head"]
summary = res.groupby("method")[cols].agg(["mean", "std"]).round(3)
summary.to_csv(EVAL_DIR / "evaluation_summary.csv")
summary

In [ ]:
order = ["FBP"] + [m for m in files]
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
for a, col, label in [(ax[0], "psnr_head", "PSNR head [dB]"), (ax[1], "ssim_head", "SSIM head")]:
    g = res.groupby("method")[col]
    a.bar(order, g.mean()[order], yerr=g.std()[order], capsize=4, color="#8aa4c8", edgecolor="#2f4b7c")
    a.set_ylabel(label); a.grid(axis="y", alpha=0.3)
    a.set_title(f"{label} – {N_VIEWS} views, {len(common)} slices")
plt.tight_layout(); plt.savefig(EVAL_DIR / "evaluation_bars.png", dpi=150); plt.show()

## Notes for the log / report
- Outputs were delivered as 8-bit PNGs → original intensity scale lost. All predictions
  (incl. FBP) are min-max scaled, rounded to 8 bit and then matched to the ground truth with a
  linear fit $a \cdot x + b$ (only brightness/contrast, not structure).
- All methods are scored on the same slices (intersection of delivered slices).
- Limitations: …